In [1]:
import os
import pandas as pd

In [2]:
def parse_excel_all_folds(excel_path):
    """
    Parse all_folds.xlsx file where each sheet represents a seed,
    and combine all fold data into a single DataFrame.
    """
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"File not found: {excel_path}")
    
    try:
        # Read all sheets
        xl = pd.ExcelFile(excel_path, engine='openpyxl')
        all_folds_data = []
        
        for sheet_name in xl.sheet_names:
            df_sheet = pd.read_excel(excel_path, sheet_name=sheet_name, engine='openpyxl')
            all_folds_data.append(df_sheet)
        
        # Combine all sheets into one DataFrame
        df_combined = pd.concat(all_folds_data, ignore_index=True)
        
        print(f"Successfully loaded data from: {excel_path}")
        print(f"Number of sheets (seeds): {len(xl.sheet_names)}")
        print(f"Combined shape: {df_combined.shape}")
        print(f"Columns: {df_combined.columns.tolist()}")
        print(f"Unique seeds: {df_combined['Seed'].nunique()}")
        print(f"Unique folds: {sorted(df_combined['Fold'].unique())}")
        
        return df_combined
    
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        raise

def merge_dataframes(F5_dir, L5_dir, basename):
    df_f5 = parse_excel_all_folds(F5_dir)
    df_l5 = parse_excel_all_folds(L5_dir)
    df_merged = pd.concat([df_f5, df_l5], ignore_index=True)
    dataset_prefix = F5_dir[3]
    output_dir = f'10S{dataset_prefix}_{basename}'
    os.makedirs(output_dir, exist_ok=True)
    print(f"Merged DataFrame shape: {df_merged.shape}")
    df_merged.to_excel(os.path.join(output_dir, f"10S{dataset_prefix}_{basename}_all_folds.xlsx"), index=False, engine='openpyxl')
    # save summary
    # Calculate summary statistics mean and std_dev
    metrics = [
            'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC',
            'Training Time (s)', 'Epochs Run', 'Avg Time/Epoch (s)',
            'Min GPU Memory (MB)', 'Max GPU Memory (MB)', 'Avg GPU Memory (MB)',
            'Min GPU Utilization (%)', 'Max GPU Utilization (%)', 'Avg GPU Utilization (%)'
        ]
    summary_df = pd.DataFrame({
        'Mean': df_merged[metrics].mean(),
        'Std Dev': df_merged[metrics].std()
    })
    with pd.ExcelWriter(os.path.join(output_dir, f"10S{dataset_prefix}_{basename}_summary.xlsx"), engine='openpyxl') as writer:
        # Sheet 1: Overall mean and std dev
        summary_df.to_excel(writer, sheet_name='Overall_Summary', index=True)
    summary_df
    return df_merged, summary_df

In [3]:
df_merged, summary_df = merge_dataframes(
    F5_dir='F5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln/F5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx',
    L5_dir='L5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln/L5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx',
    basename='tcn_model_fine_tune_mb_seqOut_sgd_ln')
df_merged

Successfully loaded data from: F5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln/F5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx
Number of sheets (seeds): 5
Combined shape: (25, 18)
Columns: ['Fold', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training Time (s)', 'Epochs Run', 'Avg Time/Epoch (s)', 'Min GPU Memory (MB)', 'Max GPU Memory (MB)', 'Avg GPU Memory (MB)', 'Min GPU Utilization (%)', 'Max GPU Utilization (%)', 'Avg GPU Utilization (%)', 'Seed', 'Variant', 'Dataset']
Unique seeds: 5
Unique folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Successfully loaded data from: L5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln/L5SR_tcn_model_fine_tune_mb_seqOut_sgd_ln_all_folds.xlsx
Number of sheets (seeds): 5
Combined shape: (25, 18)
Columns: ['Fold', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Training Time (s)', 'Epochs Run', 'Avg Time/Epoch (s)', 'Min GPU Memory (MB)', 'Max GPU Memory (MB)', 'Avg GPU Memory (MB)', 'Min GPU Utilization (%

,Fold,Accuracy,Precision,Recall,F1-Score,ROC-AUC,Training Time (s),Epochs Run,Avg Time/Epoch (s),Min GPU Memory (MB),Max GPU Memory (MB),Avg GPU Memory (MB),Min GPU Utilization (%),Max GPU Utilization (%),Avg GPU Utilization (%),Seed,Variant,Dataset
0,1,0.912322,0.873641,0.972769,0.920544,0.968108,1186.429623,13,91.263817,21199,21199,21199,74,93,89.076923,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
1,2,0.907942,0.881570,0.951589,0.915242,0.965089,1181.380710,13,90.875439,21199,21199,21199,85,93,91.076923,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
2,3,0.900435,0.865937,0.957640,0.909483,0.967422,1445.170419,16,90.323151,21201,21201,21201,90,93,91.687500,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
3,4,0.897274,0.852122,0.972012,0.908127,0.961765,1270.424467,14,90.744605,21203,21203,21203,83,95,91.571429,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
4,5,0.899644,0.869204,0.950795,0.908171,0.962973,1095.199668,12,91.266639,21205,21205,21205,85,93,90.666667,33,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
5,1,0.901659,0.871795,0.951589,0.909946,0.965126,1445.026902,16,90.314181,21207,21207,21207,80,93,90.687500,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
6,2,0.892533,0.836970,0.986384,0.905556,0.963086,1182.408323,13,90.954486,21209,21209,21209,85,93,90.538462,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
7,3,0.909917,0.877241,0.962179,0.917749,0.964237,1356.626355,15,90.441757,21209,21209,21209,86,94,91.866667,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
8,4,0.902805,0.855350,0.979576,0.913258,0.963644,1094.681856,12,91.223488,21211,21211,21211,81,93,89.333333,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit
9,5,0.895693,0.848845,0.973505,0.906911,0.959433,1182.882803,13,90.990985,21213,21213,21213,90,94,92.153846,42,tcn_model_fine_tune_mb_seqOut_sgd_ln,Reddit


In [9]:
summary_df

,Mean,Std Dev
Accuracy,0.902497,0.005914
Precision,0.863451,0.013615
Recall,0.966591,0.012073
F1-Score,0.911953,0.004617
ROC-AUC,0.964071,0.002685
Training Time (s),1249.552181,139.113588
Epochs Run,13.680000,1.596425
Avg Time/Epoch (s),91.395158,0.752878
Min GPU Memory (MB),21218.040000,12.028809
Max GPU Memory (MB),21218.040000,12.028809
